In [1]:
print(123)

123


In [4]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
import os
openai_client = OpenAI(
    api_key=os.getenv("GEMINI_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)

In [5]:
from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [7]:
assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, follow these steps:\n\n1.  **Install Ollama:**\n    *   Visit [https://ollama.com/download](https://ollama.com/download).\n    *   **macOS**: Download the `.pkg` and install it.\n    *   **Windows**: Download the `.msi` and install it.\n    *   **Linux**: Open a terminal and run:\n        ```bash\n        curl -fsSL https://ollama.com/install.sh | sh\n        ```\n\n2.  **Run a model locally:**\n    *   Open a terminal.\n    *   Type the command to download and start a model (e.g., LLaMA 3):\n        ```bash\n        ollama run llama3\n        ```\n    *   This command will download the LLaMA 3 model, start it locally, and open a chat-like interface.\n\n3.  **Test the Ollama local server:**\n    *   Run the following command in your terminal:\n        ```bash\n        curl http://localhost:11434\n        ```\n    *   You should receive a JSON response similar to `{"models": [...]}`.\n\n4.  **Install the Python client (optional):**\n    *   To interact with Ollam

In [8]:
assistant.rag("How do I run Olama locally?")

"I'm sorry, but the provided context does not contain information on how to run Olama locally."

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
)

response.choices[0].message

ChatCompletionMessage(content="That's great you found a course you're interested in!\n\nWhether you can join it right away depends on a few factors. To give you the best answer, could you please tell me:\n\n1.  **What is the name of the course?**\n2.  **Where did you discover it (e.g., on a specific website, university page, online learning platform like Coursera, edX, Udemy, etc.)?**\n\nWith that information, I can help you figure out:\n*   If enrollment is currently open.\n*   If it's a self-paced course you can start anytime.\n*   If there are specific start dates or deadlines you might have missed.\n*   How to register or join.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)

In [10]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [11]:
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the DataTalks.Club course FAQ database for information.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string"
                }
            },
            "required": ["query"]
        }
    }
}

In [12]:
response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]
)

In [13]:
call = response.choices[0].message

In [14]:
import json
tool_call = call.tool_calls[0]
args = json.loads(tool_call.function.arguments)
args

{'query': 'Can I join the course?'}

In [15]:
results=search(**args)

In [16]:
result_json = json.dumps(results, indent=2)

In [17]:
print(result_json) 

[
  {
    "id": "74eb249bbf",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "I just discovered the course. Can I still join?",
    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\u2019re still accepting submissions."
  },
  {
    "id": "69d122f12e",
    "course": "llm-zoomcamp",
    "section": "General Course-Related Questions",
    "question": "Certificate: Can I follow the course in a self-paced mode and get a certificate?",
    "answer": "No, you can only get a certificate if you finish the course with a \"live\" cohort.\n\nWe don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled."
  },
  {
    "id": "aa310de435",
    "course": "llm-zoomcamp",
    "section": "Module 1: RAG"

In [18]:
function_call_output={
    "type": "function_call_output",
    "call_id": tool_call.id,
    "Output": result_json
}

In [19]:
messages.append({
    "role": "tool",
    "name":"search",
    "tool_call_id": "function-call-12276860441250591489",
    "content": '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General course questions", "text": "Yes, you can still join the course..."}'
})

In [20]:
messages

[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 {'role': 'tool',
  'name': 'search',
  'tool_call_id': 'function-call-12276860441250591489',
  'content': '{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General course questions", "text": "Yes, you can still join the course..."}'}]

In [21]:
from openai.types.chat import (
    ChatCompletionUserMessageParam,
    ChatCompletionAssistantMessageParam,
    ChatCompletionToolMessageParam
)

messages = [
    ChatCompletionUserMessageParam(
        role="user", 
        content="I just discovered the course. Can I join now?"
    ),
    ChatCompletionAssistantMessageParam(
        role="assistant",
        tool_calls=[
            {
                "id": "function-call-12276860441250591489",
                "type": "function",
                "function": {
                    "name": "search",
                    "arguments": '{"query":"join course discovered late can I join enroll late"}'
                }
            }
        ]
    ),
    ChatCompletionToolMessageParam(
        role="tool",
        name="search",  
        tool_call_id="function-call-12276860441250591489",
        content='{"id": "74eb249bbf", "course": "llm-zoomcamp", "section": "General course questions", "text": "Yes, you can still join the course. If you want to receive a certificate, you need to submit your project..."}'
    )
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool]
)

print(response.choices[0].message.content)

Yes, you can still join the course. If you want to receive a certificate, you will need to submit your project.


In [22]:
print(response.choices[0].message)

ChatCompletionMessage(content='Yes, you can still join the course. If you want to receive a certificate, you will need to submit your project.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)


In [23]:

usage = response.usage
usage.prompt_tokens, usage.completion_tokens

(149, 25)

In [24]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(149, 25)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 3.735e-05


In [25]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

In [31]:
def make_call(call):
    args = json.loads(call.function.arguments)
    if call.function.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return result_json  

In [32]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": question},
]

response = openai_client.chat.completions.create(
    model="gemini-2.5-flash",
    messages=messages,
    tools=[search_tool],
)

messages.append(response.choices[0].message)

tool_calls = response.choices[0].message.tool_calls
if tool_calls:
    for tool_call in tool_calls:
        print("function_call:", tool_call.function.name, tool_call.function.arguments)
        call_output = make_call(tool_call)
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": call_output,
        })
    has_function_calls = True
else:
    has_function_calls = False
    print("ASSISTANT:")
    print(response.choices[0].message.content)

function_call: search {"query":"join course"}


In [34]:
def make_call(call):
    args = json.loads(call.function.arguments)
    if call.function.name == "search":
        result = search(**args)
    return json.dumps(result, indent=2)


def agent_loop(instructions, question, model="gemini-2.5-flash") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1

    while True:
        print(f"iteration #{it}...")

        response = openai_client.chat.completions.create(
            model=model,
            messages=messages,
            tools=[search_tool]
        )

        message = response.choices[0].message
        messages.append(message)

        tool_calls = message.tool_calls
        if tool_calls:
            for tool_call in tool_calls:
                print("function_call:", tool_call.function.name, tool_call.function.arguments)
                call_output = make_call(tool_call)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": call_output,
                })
        else:
            print("ASSISTANT:")
            print(message.content)
            last_answer = message.content
            break

        it += 1

    return last_answer

In [35]:
agent_loop(instructions, "How do I run Olama locally?")

iteration #1...
function_call: search {"query":"Olama locally"}
iteration #2...
ASSISTANT:
You can serve a model locally using Ollama. The course materials suggest that most local model serving tools, including Ollama, expose an OpenAI-compatible endpoint. This means you might only need to change the `base_url` in your course code to point to your local Ollama instance.

For detailed instructions on how to set up and run Ollama locally, please refer to the official Ollama website: [https://ollama.com/](https://ollama.com/).

Do you have any other questions or areas you'd like to explore?


"You can serve a model locally using Ollama. The course materials suggest that most local model serving tools, including Ollama, expose an OpenAI-compatible endpoint. This means you might only need to change the `base_url` in your course code to point to your local Ollama instance.\n\nFor detailed instructions on how to set up and run Ollama locally, please refer to the official Ollama website: [https://ollama.com/](https://ollama.com/).\n\nDo you have any other questions or areas you'd like to explore?"

In [36]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "I just discovered the course. Can I join it?")

iteration #1...
function_call: search {"query":"can I join the course late"}
function_call: search {"query":"course enrollment"}
iteration #2...
ASSISTANT:
Yes, you can still join the course. You can start whenever you want, as the videos and GitHub materials are readily available.

However, if you are aiming to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded to those who complete the course with a "live" cohort, as this involves peer-reviewing other projects, which is only possible when the course is actively running.

You can find the deadlines and start learning by checking the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp/). The deadlines are listed in the [course management platform](https://cour

'Yes, you can still join the course. You can start whenever you want, as the videos and GitHub materials are readily available.\n\nHowever, if you are aiming to receive a certificate, you need to submit your project while submissions are still being accepted. Certificates are only awarded to those who complete the course with a "live" cohort, as this involves peer-reviewing other projects, which is only possible when the course is actively running.\n\nYou can find the deadlines and start learning by checking the [LLM Zoomcamp docs](https://datatalks.club/docs/courses/llm-zoomcamp/), the [general Zoomcamp logistics docs](https://datatalks.club/docs/courses/zoomcamp-logistics/), and the [LLM Zoomcamp GitHub repository](https://github.com/DataTalksClub/llm-zoomcamp/). The deadlines are listed in the [course management platform](https://courses.datatalks.club/llm-zoomcamp-2026/).\n\nIs there anything else you\'d like to know about the course?'

In [37]:
agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I couldn't find any information about "Queen's Gambit" directly in our course materials.

The Queen's Gambit is most famously known as:
*   A popular opening in chess.
*   A critically acclaimed Netflix miniseries about a chess prodigy.

Could you clarify if you encountered this term in a specific context related to the course, or if you were thinking of something else?


'I couldn\'t find any information about "Queen\'s Gambit" directly in our course materials.\n\nThe Queen\'s Gambit is most famously known as:\n*   A popular opening in chess.\n*   A critically acclaimed Netflix miniseries about a chess prodigy.\n\nCould you clarify if you encountered this term in a specific context related to the course, or if you were thinking of something else?'

In [38]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
ASSISTANT:
I can only answer questions about the course. Is there anything about the course that you'd like to know?


"I can only answer questions about the course. Is there anything about the course that you'd like to know?"